# RL with vLLM — Interactive Demo

This notebook walks through the **GRPO training loop** used in DeepSeek-R1, Qwen3, and most modern RL-trained LLMs.

```
Prompts → vLLM (rollouts) → Reward Function → Advantages → Gradient Update → repeat
```

**vLLM's role**: it is the *rollout engine* — it generates N completions per prompt from the current policy.
The gradient update happens separately in PyTorch (e.g. TRL, OpenRLHF, veRL).

> **OpenPipe angle**: OpenPipe's core value is helping companies define reward functions for their specific tasks, then running this loop to fine-tune smaller, cheaper models.

## Cell 1 — Connect to vLLM

The vLLM pod is running Qwen2.5-7B-Instruct on spark-02.
We connect via the OpenAI-compatible API — the same interface used in production.

In [ ]:
from openai import OpenAI
import statistics, re, json

# This will be auto-updated once the pod IP is known
VLLM_URL = "VLLM_POD_URL_PLACEHOLDER"
MODEL    = "Qwen/Qwen2.5-7B-Instruct"

client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="none")

# Verify connection
models = client.models.list()
print(f"Connected! Available models: {[m.id for m in models.data]}")

## Cell 2 — The RL Training Loop (diagram)

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────────┐
│  Prompts    │────▶│  vLLM            │────▶│  G completions      │
│  (tasks)    │     │  n=G, temp>0     │     │  per prompt         │
└─────────────┘     └──────────────────┘     └──────────┬──────────┘
                                                         │
                    ┌────────────────────────────────────▼──────────┐
                    │  Reward Function  →  [r1, r2, r3, r4]         │
                    │  You define what "good" means here            │
                    └────────────────────────────────────┬──────────┘
                                                         │
                    ┌────────────────────────────────────▼──────────┐
                    │  GRPO Advantage = (ri - mean) / std           │
                    │  No value network — group normalizes itself    │
                    └────────────────────────────────────┬──────────┘
                                                         │
                    ┌────────────────────────────────────▼──────────┐
                    │  loss = -mean(advantage_i × log_prob_i)       │
                    │  High advantage → reinforce  ↑                │
                    │  Low advantage  → suppress   ↓                │
                    └───────────────────────────────────────────────┘
                                          │
                       weights updated ───┘  (then synced back to vLLM)
```

**Why temperature > 0 matters**: RL requires *exploration* — the model must sometimes output
suboptimal completions to discover that they're bad. Greedy decoding (temp=0) would always
pick the same output, giving the optimizer nothing to learn from.

## Cell 3 — Step 1: Define Tasks

Simple math problems with verifiable answers — the easiest reward signal to start with.
DeepSeek-R1 was bootstrapped exactly this way: math + code where correctness is binary.

In [ ]:
TASKS = [
    {"prompt": "What is 47 + 38?",   "answer": 85},
    {"prompt": "What is 9 × 8?",     "answer": 72},
    {"prompt": "What is 144 ÷ 12?",  "answer": 12},
    {"prompt": "What is 25 + 75?",   "answer": 100},
]

SYSTEM_PROMPT = (
    "You are a math assistant. Answer ONLY with the final number, "
    "no explanation. Example: 42"
)

G = 4          # Group size: completions per prompt
TEMP = 0.8     # Must be > 0 for exploration

print(f"Tasks: {len(TASKS)}, G={G}, temperature={TEMP}")
print("Ready to generate rollouts.")

## Cell 4 — Step 2: Generate Rollouts with vLLM

`n=G` in the API call tells vLLM to generate G completions for the same prompt in one batch.
This is the core of vLLM's value in RL: it handles the batching efficiently using PagedAttention,
so you can generate thousands of rollouts without OOM.

In [ ]:
def generate_rollouts(task):
    """Ask vLLM for G completions of this task. This IS the rollout step."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": task["prompt"]},
        ],
        n=G,
        temperature=TEMP,
        max_tokens=32,
    )
    return [
        {"text": c.message.content.strip(), "answer": task["answer"]}
        for c in response.choices
    ]

# Try it on one task
sample_task = TASKS[0]
completions = generate_rollouts(sample_task)

print(f"Prompt: {sample_task['prompt']}  (answer: {sample_task['answer']})")
print(f"\nvLLM returned {len(completions)} rollouts:")
for i, c in enumerate(completions):
    print(f"  [{i+1}] '{c['text']}'")

## Cell 5 — Step 3: Reward Function (YOUR TURN)

**This is the most important cell.** The reward function defines what "good" means.

This is also OpenPipe's core product — helping companies operationalize reward signals
for their specific use cases (structured output, tone, accuracy, code correctness, etc.).

Implement `compute_reward` below. Then run the cell and see how it scores the rollouts above.

Things to consider:
- Should `"The answer is 85"` score the same as `"85"`? 
- Binary (0/1) or partial credit?
- What about `"85.0"` or `"85!"`?

In [ ]:
def compute_reward(completion):
    """
    TODO: implement this.
    
    Args:
        completion["text"]   -- model output string, e.g. "85" or "the answer is 85"
        completion["answer"] -- correct integer, e.g. 85
    
    Returns: float reward
    """
    text   = completion["text"]
    answer = completion["answer"]
    
    # --- your implementation here ---
    raise NotImplementedError("Implement compute_reward!")


# Test it
print("Testing your reward function on the sample rollouts:")
for c in completions:
    r = compute_reward(c)
    print(f"  output='{c['text']:>15}'  reward={r:.2f}")

## Cell 6 — Step 4: GRPO Advantage Computation

GRPO's key insight: instead of training a separate value network (like PPO does),
normalize rewards *within the group of G completions*.

```
advantage_i = (reward_i - mean(rewards)) / std(rewards)
```

This is cheap and stable. DeepSeek used this exact trick with G=8 to train R1.

In [ ]:
def compute_advantages(rewards):
    if len(set(rewards)) == 1:
        return [0.0] * len(rewards)  # all same reward → no signal
    mean_r = statistics.mean(rewards)
    std_r  = statistics.stdev(rewards)
    return [(r - mean_r) / (std_r + 1e-8) for r in rewards]

# Score and compute advantages for sample task
rewards    = [compute_reward(c) for c in completions]
advantages = compute_advantages(rewards)

print(f"Prompt: {sample_task['prompt']}")
print(f"{'Output':>20}  {'Reward':>8}  {'Advantage':>10}  Signal")
print("-" * 60)
for c, r, a in zip(completions, rewards, advantages):
    signal = "↑ reinforce" if a > 0.1 else ("↓ suppress" if a < -0.1 else "≈ neutral")
    print(f"{c['text']:>20}  {r:>8.2f}  {a:>+10.2f}  {signal}")

## Cell 7 — Full Loop Across All Tasks

In [ ]:
all_rewards = []

for task in TASKS:
    print(f"\n{'─'*55}")
    print(f"  {task['prompt']}  (answer: {task['answer']})")
    print(f"{'─'*55}")
    
    completions = generate_rollouts(task)
    rewards     = [compute_reward(c) for c in completions]
    advantages  = compute_advantages(rewards)
    
    for c, r, a in zip(completions, rewards, advantages):
        sig = "↑" if a > 0.1 else ("↓" if a < -0.1 else "≈")
        print(f"  {c['text']:>12}  r={r:.2f}  adv={a:+.2f} {sig}")
    
    all_rewards.extend(rewards)

print(f"\n{'='*55}")
print(f"  Mean reward across all tasks: {statistics.mean(all_rewards):.3f}")
print(f"  (1.0 = perfect, 0.0 = all wrong)")
print(f"{'='*55}")

## Cell 8 — What the Gradient Update Would Look Like

In real RL training (TRL/OpenRLHF/veRL), after collecting rollouts and advantages, the update is:

```python
# For each (prompt, completion, advantage) triple:
log_probs = policy_model(prompt + completion)  # forward pass
loss = -(advantage * log_probs).mean()         # REINFORCE objective
loss.backward()                                 # gradient
optimizer.step()                                # weight update

# Then sync updated weights back to vLLM:
vllm_engine.update_weights(policy_model.state_dict())
```

The **weight sync** (`update_weights`) is what connects the trainer to vLLM.
Modern frameworks like **veRL** (from ByteDance) and **OpenRLHF** handle this automatically.

**Why OpenPipe cares about this:**  
The reward function you wrote in Cell 5 is what drives everything above.
For a coding assistant, it might be "does this code pass the unit tests?"  
For a customer support bot, it might be "does this response follow the tone rubric?"  
Getting that signal right — and scalably — is the entire product.

## Cell 9 — Experiment: What Happens with Different Reward Shapes?

Try modifying `compute_reward` to use these different strategies and re-run Cell 6:

**Binary (hard)**:
```python
return 1.0 if str(answer) in text else 0.0
```

**Partial credit (soft)**:
```python
nums = re.findall(r'-?\d+\.?\d*', text)
if not nums: return 0.0
closest = min(nums, key=lambda x: abs(float(x) - answer))
error = abs(float(closest) - answer)
return max(0.0, 1.0 - error / answer)
```

**Format + correctness** (OpenPipe-style):
```python
# Reward exact format match more than embedded answer
if text.strip() == str(answer): return 1.0         # perfect format + correct
if str(answer) in text:         return 0.5         # correct but messy format  
return 0.0                                          # wrong
```

Notice how the choice of reward shape changes which completions get reinforced vs suppressed.